# PFR 03 — Lead screen: which catalog features predict the perp return AFTER t?

Protocol: `discussions/Pairwise_Feature_Regression/007_claude_lead_screen_preregistration.md`.

For each PFR 01 catalog feature $X_t$ and $h\in\{5,15,60\}$ min: partial IC of $X_t$ with the Deribit perp forward return $y_{t,h}$ after removing the current perp return. Discovery = first 60% of UTC days, confirmation = last 40%. Whole-day bootstrap. A cell is **confirmed** if discovery passes Bonferroni (features × horizons, per asset) and confirmation has the same sign at one-sided 5%. Decile long–short (bp) in confirmation is compared with a 12 bp round trip.

In [1]:
import os,sys,json,importlib,unittest,importlib.util
from pathlib import Path
import numpy as np, pandas as pd
from IPython.display import display
from dotenv import load_dotenv
ROOT=next((p for p in [Path.cwd().resolve(),*Path.cwd().resolve().parents] if (p/'src'/'data_engineering').is_dir()),None)
assert ROOT is not None,'Run inside the repository'
sys.path.insert(0,str(ROOT/'src'));load_dotenv(ROOT/'.env')
from data_engineering.modeling import pairwise_feature_lead as pl
from data_engineering.modeling import pairwise_feature_regression as pfr
for m in (pfr,pl):importlib.reload(m)
assert pl.VERSION=='PFR03-0.1.0',pl.VERSION
cfg=pl.Specification()
SYNTHETIC=os.getenv('PFR03_SYNTHETIC','0')=='1'
OUT=ROOT/'notebooks'/'Pairwise_Feature_Regression'/'03'
if SYNTHETIC:
    from tempfile import TemporaryDirectory
    tmp=TemporaryDirectory(prefix='pfr03-');OUT=Path(tmp.name)/'03'
OUT.mkdir(parents=True,exist_ok=True)
# in-kernel tests (Windows/Conda-safe, as PFR 01)
spec_t=importlib.util.spec_from_file_location('_pfr03_tests',ROOT/'tests'/'test_pairwise_feature_lead.py')
tm=importlib.util.module_from_spec(spec_t);spec_t.loader.exec_module(tm)
import pytest
rc=pytest.main(['-q',str(ROOT/'tests'/'test_pairwise_feature_lead.py')])
assert rc==0,'PFR03 tests failed'
print('PFR03',pl.VERSION,'synthetic:',SYNTHETIC,'output:',OUT)

.......                                                                                                          [100%]
7 passed in 0.51s
PFR03 PFR03-0.1.0 synthetic: False output: D:\Projects\deribit_market_data\notebooks\Pairwise_Feature_Regression\03


In [2]:
ASSET_DATA={}
if SYNTHETIC:
    for i,(a,b) in enumerate((('BTC',.12),('ETH',.0))):
        f,t,r=pl.synthetic_states(n_days=28,seed=10+i,beta_lead=b,bounce=-.1)
        ASSET_DATA[a]=(f,t,r)
    source='SYNTHETIC'
else:
    import duckdb
    from data_engineering.catalog.database import RESEARCH_DATABASE
    from data_engineering.modeling.full_information import FEATURE_SPECS
    q=lambda n:'"'+n.replace('"','""')+'"'
    cols=list(dict.fromkeys(['underlying','state_time','expiry_date','utc_minute_of_day','session_progress_fraction',
        'settlement_locked_fraction','is_weekend','minutes_to_expiry','in_settlement_window','perp_integrity_ok',
        'microstructure_features_ready','instability_features_ready','option_atm_quote_ok','option_shape_quote_ok',
        'fit_ok','option_snapshot_ready']+[s.column for s in FEATURE_SPECS]+[e[1] for e in pfr.EXTRAS]))
    with duckdb.connect(str(RESEARCH_DATABASE),read_only=True) as con:
        con.execute("SET TimeZone='UTC'")
        fields=set(con.execute('DESCRIBE SELECT * FROM time_series_feature_state').df().column_name)
        want=[c for c in cols if c in fields]
        sql='SELECT '+', '.join(map(q,want))+' FROM time_series_feature_state WHERE underlying IN ('+','.join('?'*len(cfg.assets))+') ORDER BY underlying, state_time'
        states=con.execute(sql,list(cfg.assets)).df()
    states['state_time']=pd.to_datetime(states.state_time,utc=True)
    assert not states.duplicated(['underlying','state_time']).any()
    last_full=states.state_time.max().floor('D')
    states=states[states.state_time<last_full]                      # complete UTC days only
    print('States:',len(states),states.state_time.min(),'->',states.state_time.max())
    for a,part in states.groupby('underlying'):
        part=part.reset_index(drop=True)
        feats,inv=pfr.model_feature_frame(part)
        ASSET_DATA[a]=(feats,part.state_time,feats['perp_ret_5m'])   # integrity-gated perp return is the target
    source=f'research.duckdb time_series_feature_state, {cfg.assets}'
print({a:len(v[0]) for a,v in ASSET_DATA.items()})

States: 27264 2026-09-19 08:00:00+00:00 -> 2026-10-12 23:55:00+00:00
{'BTC': 6816, 'ETH': 6816, 'SOL': 6816, 'XRP': 6816}


In [3]:
tables=[];metas={}
for a,(f,t,r) in ASSET_DATA.items():
    tab,meta=pl.screen_asset(f,t,r,cfg)
    tables.append(tab.assign(asset=a));metas[a]=meta;print(a,meta)
table=pd.concat(tables,ignore_index=True)
cols=['asset','feature','horizon_min','disc_raw_ic','disc_partial_ic','conf_raw_ic','conf_partial_ic','conf_partial_lo','conf_partial_hi',
      'conf_partial_n','discovery_pass','confirmed','conf_long_short_bp','covers_cost']
print('Confirmed cells:')
display(table[table.confirmed][cols].sort_values(['horizon_min','asset']).round(4))
print('Discovery passes that failed confirmation:')
display(table[table.discovery_pass&~table.confirmed][cols].round(4))

BTC {'days': 24, 'discovery_days': ['2026-09-19', '2026-10-02'], 'confirmation_days': ['2026-10-03', '2026-10-12'], 'n_features': 81, 'tests_per_asset': 243, 'discovery_two_sided_alpha_per_test': 0.000205761316872428}
ETH {'days': 24, 'discovery_days': ['2026-09-19', '2026-10-02'], 'confirmation_days': ['2026-10-03', '2026-10-12'], 'n_features': 81, 'tests_per_asset': 243, 'discovery_two_sided_alpha_per_test': 0.000205761316872428}
SOL {'days': 24, 'discovery_days': ['2026-09-19', '2026-10-02'], 'confirmation_days': ['2026-10-03', '2026-10-12'], 'n_features': 81, 'tests_per_asset': 243, 'discovery_two_sided_alpha_per_test': 0.000205761316872428}
XRP {'days': 24, 'discovery_days': ['2026-09-19', '2026-10-02'], 'confirmation_days': ['2026-10-03', '2026-10-12'], 'n_features': 81, 'tests_per_asset': 243, 'discovery_two_sided_alpha_per_test': 0.000205761316872428}
Confirmed cells:


,asset,feature,horizon_min,disc_raw_ic,disc_partial_ic,conf_raw_ic,conf_partial_ic,conf_partial_lo,conf_partial_hi,conf_partial_n,discovery_pass,confirmed,conf_long_short_bp,covers_cost


Discovery passes that failed confirmation:


,asset,feature,horizon_min,disc_raw_ic,disc_partial_ic,conf_raw_ic,conf_partial_ic,conf_partial_lo,conf_partial_hi,conf_partial_n,discovery_pass,confirmed,conf_long_short_bp,covers_cost
171,BTC,tfi,60,-0.0413,-0.0361,-0.0061,-0.0068,-0.0245,0.0150,2220,True,False,0.8762,False
172,BTC,buy_share,60,-0.0413,-0.0361,-0.0061,-0.0068,-0.0245,0.0150,2220,True,False,0.8762,False
186,BTC,obi_20_change,60,0.0185,0.0204,0.0070,0.0106,-0.0047,0.0286,2220,True,False,0.8628,False
749,XRP,log_spread_bps,5,-0.0269,-0.0270,0.0376,0.0346,0.0013,0.0599,2714,True,False,NaN,False
810,XRP,ret_15m,15,-0.0569,-0.0503,0.0263,0.0544,-0.0241,0.1108,2567,True,False,-1.3188,False


In [4]:
print('Largest |confirmation partial IC| per asset/horizon (descriptive, not a selection rule):')
t2=table[table.enough_data].copy();t2['abs']=t2.conf_partial_ic.abs()
display(t2.sort_values('abs',ascending=False).groupby(['asset','horizon_min']).head(5)[cols].sort_values(['asset','horizon_min']).round(4))
summ=pl.summary(table,cfg);display(summ)
dec=pl.decision(table,cfg,synthetic=SYNTHETIC);print(json.dumps(dec,indent=2,default=str))

Largest |confirmation partial IC| per asset/horizon (descriptive, not a selection rule):


,asset,feature,horizon_min,disc_raw_ic,disc_partial_ic,conf_raw_ic,conf_partial_ic,conf_partial_lo,conf_partial_hi,conf_partial_n,discovery_pass,confirmed,conf_long_short_bp,covers_cost
79,BTC,return_5m,5,-0.0301,-0.0054,-0.0247,-0.0608,-0.1470,0.0453,2712,False,False,0.4348,False
63,BTC,bf25_d5,5,-0.0678,-0.0700,-0.0376,-0.0464,-0.0995,0.0103,1596,False,False,0.9543,False
47,BTC,bf_25d,5,-0.0117,-0.0143,-0.0464,-0.0464,-0.0728,-0.0147,1691,False,False,2.0014,False
28,BTC,obi_20_min,5,0.0210,0.0204,0.0466,0.0462,0.0065,0.0817,2712,False,False,1.0141,False
29,BTC,obi_20_max,5,0.0044,0.0080,-0.0414,-0.0398,-0.0509,-0.0256,2712,False,False,-0.5720,False
142,BTC,curv025_d60,15,0.0028,0.0021,-0.0706,-0.0722,-0.1088,0.0031,854,False,False,-0.8006,False
128,BTC,bf_25d,15,0.0082,0.0055,-0.0645,-0.0714,-0.1085,-0.0287,1612,False,False,-2.8529,False
133,BTC,atm_iv_d30,15,0.0414,0.0406,-0.0688,-0.0680,-0.1324,0.0232,2298,False,False,-1.6652,False
155,BTC,weekend,15,0.0069,0.0081,0.0601,0.0586,0.0155,0.0973,2608,False,False,0.8527,False
134,BTC,atm_iv_d60,15,0.0406,0.0418,-0.0546,-0.0583,-0.1174,0.0321,2247,False,False,-1.9279,False


,feature,horizon_min,assets,signs,mean_conf_partial_ic,mean_long_short_bp,any_covers_cost


{
  "verdict": "NO_DIRECTIONAL_FEATURE",
  "n_confirmed_cells": 0,
  "worth_looking_at": [],
  "covers_cost_cells": [],
  "status": "HISTORICAL_EXPLORATORY",
  "prospective_confirmation": false
}


In [5]:
table.to_csv(OUT/'03_lead_screen.csv',index=False);summ.to_csv(OUT/'03_cross_asset_summary.csv',index=False)
(OUT/'03_decision.json').write_text(json.dumps(dec,indent=2,default=str),encoding='utf-8')
(OUT/'03_manifest.json').write_text(json.dumps({'module':'PFR03','registration':pl.registration(),'source':source,
    'synthetic':SYNTHETIC,'split':metas},indent=2,default=str),encoding='utf-8')
print('Saved 4 outputs to',OUT)

Saved 4 outputs to D:\Projects\deribit_market_data\notebooks\Pairwise_Feature_Regression\03
